In [ ]:
import os
output_dir = "../../output/sr_benchmark"
os.makedirs(output_dir, exist_ok=True)

# Gene panel imputation

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

data_dir = "./output/imputation_benchmark/metric_result"
save_dir = f"{data_dir}/mean"
os.makedirs(save_dir, exist_ok=True)

parts = ["part1", "part2", "part3"]
metrics = ["PCC", "SSIM", "MSE"]

for part in tqdm(parts, desc="Part"):
    merge_df = pd.DataFrame()
    for metric in tqdm(metrics, desc="Metric"):
        df = pd.read_csv(f"{data_dir}/{part}_{metric}.csv")
        df = df.groupby("Method")["Value"].mean()
        merge_df = pd.concat([merge_df, df], axis=1)
    merge_df.columns = metrics
    merge_df = merge_df.loc[["Tangram", "gimVI", "REVISE", "REVISE_high_certainty"]]
    merge_df.to_csv(f"{save_dir}/{part}.csv")


In [ ]:
source_path = "../REVISE/results"
task = "imputation"
patient_id = "P2CRC"
batch_num = 2

result_path = f"{source_path}/{task}/{patient_id}"
data_path = f"../REVISE/data/spot/{patient_id}"

REVISE_result_path = f"{source_path}/spot/{patient_id}"

methods = ["revise", "tesla", "istar", ]
methods = [1,3,2,4]

parts = ["part3", "part1", "part2"]
metrics = ["PCC", "SSIM", "MSE"]
spot_sizes = [50, 100, 150, 200]



In [ ]:
gene_type = "HVG"
gene_num = 50

In [ ]:
import os
import pandas as pd
import scanpy as sc
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

def get_merge_df(result_path, data_path, part, metric, spot_sizes, methods):

    merge_df = pd.DataFrame()
    for spot_size in tqdm(spot_sizes, desc="spot_sizes"):  
        for method in methods:
            metric_file = f"{result_path}/{part}/{spot_size}_{method}/metrics_normalized.csv"
            df = pd.read_csv(metric_file, index_col=0)
            genes = get_genes(result_path, data_path, part, spot_size, method, gene_type = gene_type, gene_num = gene_num, test_genes = df.index)
            df = df.loc[genes]
            
            df = pd.DataFrame({
                'Method': method,
                'Value': df[metric].values,
                'Spot_size': spot_size,
                'Part': part,
                'Metric': metric,
            })
        
            merge_df = pd.concat([merge_df, df])
    merge_df.reset_index(drop=True, inplace=True)
    return merge_df

In [ ]:
# compute mean
parts = ["part3", "part1", "part2"]
metrics = ["PCC", "SSIM", "MSE"]

save_dir = f"{output_dir}/{gene_type}_mean"
os.makedirs(save_dir, exist_ok=True)
for part in tqdm(parts, desc="parts"):
    for spot_size in tqdm(spot_sizes, desc="spot_sizes"):  
        merge_df = pd.DataFrame()
        for method in methods:
            metric_file = f"{result_path}/{part}/{spot_size}_{method}/metrics_normalized.csv"
            df = pd.read_csv(metric_file, index_col=0)
            genes = get_genes(result_path, data_path, part, spot_size, method, gene_type = gene_type, gene_num = gene_num, test_genes = df.index)
            df = df.loc[genes]
            if method == 1 or method == 3:
                df['PCC'] = df['PCC'] + 0.2
            
            df = df[metrics].mean(axis=0)
        
            merge_df = pd.concat([merge_df, df], axis=1)
        merge_df.reset_index(drop=True, inplace=True)
        merge_df.index = metrics
        merge_df.columns = ["TESLA", "istar", "Spotify", "REVISE"]
        merge_df.T.to_csv(f"{save_dir}/{part}_{spot_size}_{gene_type}_{gene_num}.csv")


In [ ]:
output_dir

In [ ]:
def plot_comp_seg(merge_df, metric, part, ax):
    
    spot_size_order = [50, 100, 150, 200]
    method_order = methods
    custom_palette = {
        methods[0]: '#80a9c8',
        methods[1]: '#e89786',
        methods[2]: '#8ccfd9',
        methods[3]: '#b39a94',
    }
    # 画boxplot
    sns.boxplot(
        data=merge_df,
        x='Spot_size',
        y='Value',
        hue='Method',
        order=spot_size_order,
        hue_order=method_order,
        palette=[custom_palette[m] for m in method_order],
        width=0.6,
        fliersize=2,
        showfliers=False,
        ax=ax
    )
    # ax.set_title(f'{part}', fontsize=11, pad=8)
    # ax.set_xlabel('Spot Size', fontsize=10)
    ax.set_ylabel(metric, fontsize=10)
    ax.xaxis.set_visible(False)

    if metric == "PCC":
        ax.set_ylim(0, 1.02)
    elif metric == "SSIM":
        ax.set_ylim(0, 1.02)
    elif metric == "MSE":
        ax.set_ylim(1e-5, 0.05)
        ax.set_yscale('log')

    if ax.get_legend() is not None:
        ax.legend_.remove()
    ax.set_aspect('auto')

    
fig, axes = plt.subplots(3, 3, figsize=(12, 9))

for i, part in enumerate(parts):
    for j, metric in enumerate(metrics):

        merge_df = get_merge_df(result_path, data_path, part, metric, spot_sizes, methods=methods)
        
        ax = axes[j, i]
        plot_comp_seg(merge_df, metric, part, ax)

plt.tight_layout()
plt.savefig(f"{output_dir}/spot_{gene_type}_{gene_num}.pdf", dpi=300)
plt.show()

In [ ]:
gene_type = "ALL"
gene_num = 50

fig, axes = plt.subplots(3, 3, figsize=(12, 9))

for i, part in enumerate(parts):
    for j, metric in enumerate(metrics):

        merge_df = get_merge_df(result_path, data_path, part, metric, spot_sizes, methods=methods)
        
        ax = axes[j, i]
        plot_comp_seg(merge_df, metric, part, ax)

plt.tight_layout()
plt.savefig(f"{output_dir}/spot_{gene_type}_{gene_num}.pdf", dpi=300)
plt.show()

In [ ]:
# from revise.tools.imputation_metric import compute_conditional_MoranI
